In [ ]:
# ── Notebook parameters ─────────────────────────────────────────
# Defaults for interactive use. extract_notebook_figures.py will
# override these via a cell injected right below this one.
#
# Add notebook-level globals here, e.g.:
# Analyzes a CelebA FPCA checkpoint. Produce one first with
#   python fpca.py +fpca_experiment=celeba_eulerian
# The run directory name is not predictable: `wandb-<run id>` with wandb enabled,
# otherwise a timestamp. List outputs/checkpoints/, pick your run, and use the
# highest step_*.pt (or best.pt) inside it.
CHECKPOINT_DIR  = "../outputs/checkpoints/XXX"
CHECKPOINT_FILE = "step_XXX.pt"

# Number of leading prefix atoms shown in the basis comparison.
N_SHOW_LINEAR = 12

# Dataset and general visualizations
N_VIS = 64
N_PEEK = 6

# Basis visualizations
MOSAIC_SIDE = 11         # mosaic of the first MOSAIC_SIDE² prefix atoms
N_VIS_ATOM = 128
TILE_SIZE = 0.55
ZOOM_ATOM_IDX = 11       # 0-based prefix position of the zoomed atom
ZOOM_LEFT = 1 / 2.0 - 0.1
ZOOM_RIGHT = 5 / 6.0
ZOOM_DOWN = 1 / 6.0 + 0.05
ZOOM_UP = 1 / 2.0 + 0.15
ZOOM_FULL_RES = 228
ZOOM_RES = 6
MIXING_LAYERS = None   # 0-based layer indices to plot; None = all layers
N_VIS_U = 64

# Reconstruction and spectral diagnostics
RECON_ATOM_BUDGETS = [3, 27, 75, 243, 1083, 2523]   # prefix lengths
N_VAL = 4
K_SHOW = 12
N_OOD_SHOW = 4
N_SPEC = 300
SPECTRAL_ATOMS = 243
N_VIOLIN = 10



# Functional PCA — Learned Isometry vs. Baselines

This notebook accompanies `fpca.py` for **Fourier-based image checkpoints**.
Every image-specific quantity (`CHANNELS_DIM`, `COORDS_DIM`, resolution,
drop fraction, ...) is read from the saved Hydra `config.yaml`, so it works
for grayscale (`C=1`) and RGB (`C=3`) Fourier checkpoints.

All visualisation goes through the canonical helpers in
`notebook_helpers.py` — `field_to_img`, `to_rgb_image`, `render`,
`show_field`, `plot_atom_mosaic`, `scatter_signal`, `visualize_mixing_layers`
— so per-notebook copies of the same logic are not allowed; please edit
the helpers if you need to change behaviour. The canonical visualisation
grid is always `SquareSampler(stratified=True, add_noise=False).sample(N_VIS)`
(ij layout: `coords[i*N+j] = (x_i, y_j)`).

For datasets that pre-compute their own pixel grid in *yx* layout (CelebA's
`_full_coords` has y outer, x inner) we permute the stored signals into ij
layout via `yx_to_ij` so the rest of the notebook can use the unified
helpers without flipping conventions.

The dictionary is only probed through its ordered prefix $e_1,\dots,e_K$
(atoms by descending PMF). The isometry is a stack of `num_layers` Cayley
rotations built from the first $R=$ `rank` atoms of that prefix ($R$ is the
top-level `rank` key of the run config), and learned atom $k$ is row $k$ of the
pushed prefix $T(e_1,\dots,e_K)$, $K \ge R$. The isometry preserves the Gram
matrix $\langle e_i, e_j\rangle$ of the prefix, which need not be orthonormal
(Fourier happens to be), so reconstructions and energy curves use orthogonal
projections onto the span of the first $k$ atoms rather than raw inner
products. Every section evaluates the shortest prefix that covers what it shows.

Sections:
1. Setup
2. Canonical grid & corpus peek
3. **Pushforward of Fourier atoms** (loaded checkpoint)
4. **Mixing layers: functions $u_r$ and causal attention $A$** (loaded checkpoint)
5. Mean function
6. Reconstruction with the learned basis
7. Baselines (PCA gold + interpolated PCA)
8. PSNR comparison

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, math
sys.path.insert(0, "..")

import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from hydra.utils import instantiate

from infidictionary.datasets import IrregularDataset
from infidictionary.dictionaries import InfiDictionary
from infidictionary.networks import NeuralField
from infidictionary.neural_isometries import NeuralIsometry
from infidictionary.utils import (
    norm2, nested_projection_energies, prefix_weighted_energy, project_onto_span,
)
from infidictionary.domain_samplers import SquareSampler

from notebook_helpers import (
    field_to_img, to_rgb_image, render, show_field, visualize_mixing_layers,
    yx_to_ij, coords_yx_to_ij,
    psnr, plot_recon_grid, psnr_matrix, plot_psnr_line_chart,
    plot_atoms_comparison,
    plot_atom_mosaic, scatter_signal,
    fetch_validation_signals, build_corpus, pca_reconstruct,
    plot_spectral_compactness,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def reconstruct_in_basis(
    functions: torch.Tensor, mean: torch.Tensor, basis: torch.Tensor,
) -> torch.Tensor:
    """Orthogonal projection of centred ``functions`` (B, N, C) onto span(``basis``), plus the mean.

    ``basis`` (k, N, C) need not be orthonormal: ``project_onto_span`` solves
    with its Gram matrix. For an orthonormal prefix (Fourier happens to be one)
    this is Σ_j ⟨f - μ, e_j⟩ e_j. With ``basis = learned_prefix(coords, k)`` it
    projects onto span(T e_1, …, T e_k).
    """
    projected, _ = project_onto_span(functions - mean.unsqueeze(0), basis)
    return projected + mean.unsqueeze(0)

def print_threshold_table(cum_fourier_mean, cum_learned_mean, fracs=(0.50, 0.75, 0.90)):
    """Atoms needed to reach each fraction of ‖f - μ‖², from mean cumulative-fraction curves."""
    def need(cum, frac):
        n = int((cum < frac).sum()) + 1
        return n if n <= len(cum) else None   # None: not reached within the prefix
    print(f"\n{'Threshold':>10}  {'Fourier':>10}  {'Learned':>10}  {'gain':>8}")
    for frac in fracs:
        nf, nl = need(cum_fourier_mean, frac), need(cum_learned_mean, frac)
        cells = [f"{n:>9d}" if n else f"{'>' + str(len(cum_fourier_mean)):>9}" for n in (nf, nl)]
        gain = f"{(nf - nl) / nf * 100:>+7.1f}%" if nf and nl else f"{'n/a':>8}"
        print(f"{frac*100:>9.0f}%  {cells[0]}  {cells[1]}  {gain}")

def prefix_mosaic_indices(side: int) -> torch.Tensor:
    """Layout indices for ``plot_atom_mosaic`` over the first ``side²`` prefix atoms.

    Rows hold consecutive prefix positions, starting from the bottom row; the
    1-based prefix position of a tile is its row label plus its column label.
    """
    pos = torch.arange(side * side)
    return torch.stack([(pos // side) * side, pos % side + 1, torch.zeros_like(pos)], dim=1)

## 1. Setup — load checkpoint

Set `CHECKPOINT_DIR` and `CHECKPOINT_FILE`. Resolution, channel count, drop
fraction, dictionary parameters, etc. are *all* read from `config.yaml`.

In [ ]:


with open(f"{CHECKPOINT_DIR}/config.yaml") as f:
    conf = OmegaConf.create(yaml.safe_load(f))

neural_isometry: NeuralIsometry      = instantiate(conf.neural_isometry).to(device)
mean_function:   NeuralField         = instantiate(conf.mean_function).to(device)
initial_dictionary: InfiDictionary   = instantiate(conf.initial_dictionary).to(device)
function_generator: IrregularDataset = instantiate(conf.function_generator)

ckpt = torch.load(f"{CHECKPOINT_DIR}/{CHECKPOINT_FILE}", map_location=device, weights_only=False)
neural_isometry.load_state_dict(ckpt["models"]["neural_isometry"])
mean_function.load_state_dict(ckpt["models"]["mean_function"])
if "initial_dictionary" in ckpt["models"]:
    initial_dictionary.load_state_dict(ckpt["models"]["initial_dictionary"])

neural_isometry.eval()
mean_function.eval()

# Ordered prefix the isometry was trained on: the top-`num_atoms` atoms
# e_1..e_K (descending PMF). Learned atom k is row k of the pushed prefix. The
# rotations are built from its first R rows; R is the top-level `rank` key of
# the run config and is passed to every pushforward (it is not a constructor
# argument), which needs K >= R. By prefix faithfulness the first K pushed rows
# do not depend on how many further atoms are appended.
NUM_ATOMS   = int(conf.num_atoms)
RANK        = int(conf["rank"])
PREFIX_PMFS = initial_dictionary.get_prefix_pmfs(NUM_ATOMS).to(device)

def raw_prefix(coords, num_atoms):
    """The first ``num_atoms`` Fourier atoms e_1..e_K at ``coords``: (K, N, C)."""
    return initial_dictionary.get_prefix(coords, num_atoms)

def learned_prefix(coords, num_atoms):
    """The first ``num_atoms`` learned atoms, rows of T_R(e_1..e_K) at ``coords``: (K, N, C).

    Pushes a prefix of length max(K, R), since the pushforward needs K >= R,
    and keeps the first K rows.
    """
    lad = torch.zeros(coords.shape[0], device=coords.device)
    prefix = initial_dictionary.get_prefix(coords, max(num_atoms, RANK))
    _, _, pushed = neural_isometry.pushforward(coords, lad, prefix, rank=RANK)
    return pushed[:num_atoms]

COORDS_DIM   = int(conf.initial_dictionary.get("domain_dim", 2))
CHANNELS_DIM = int(conf.initial_dictionary.get("num_channels", 1))

print(f"Loaded   : {CHECKPOINT_DIR}/{CHECKPOINT_FILE}  (epoch {ckpt['epoch']})")
print(f"Metric   : {ckpt.get('metric', float('nan')):.4f}  |  best: {ckpt.get('best_metric', float('nan')):.4f}")
print(f"Generator: {type(function_generator).__name__}")
print(f"Domain   : d={COORDS_DIM}   |   channels: C={CHANNELS_DIM}")
print(f"Prefix   : K={NUM_ATOMS} atoms, rank R={RANK}, layers={getattr(neural_isometry, 'num_layers', '-')}")

## 2. Canonical grid & corpus peek

The visualisation grid `coords_full` always comes from
`SquareSampler(stratified=True, add_noise=False)` (ij layout). For
generators that store data on a *yx* pixel grid (e.g. CelebA), we set
`HAS_YX_DATA = True`; the helper `to_ij_signal(s)` converts a stored
`(M, N, C)` corpus into ij layout for display and PCA, and
`val_signals_ij` holds the held-out signals on the *same* grid as
`coords_full`.

In [ ]:
coords_full = SquareSampler(stratified=True, add_noise=False).sample(N_VIS).to(device)
N_FULL      = coords_full.shape[0]
logabsdet_full = torch.zeros(N_FULL, device=device)

# Detect a yx-layout pre-computed grid (CelebA-style); if present, build a
# converter so we can reuse the stored corpus on the ij visualisation grid.
HAS_YX_DATA = hasattr(function_generator, "_full_coords")
if HAS_YX_DATA:
    coords_yx = function_generator._full_coords.to(device)
    n_yx      = int(round(math.sqrt(coords_yx.shape[0])))
    assert n_yx == N_VIS, (
        f"`function_generator._full_coords` is {n_yx}x{n_yx} but N_VIS={N_VIS}; "
        "set N_VIS to match the dataset resolution."
    )
    def to_ij_signal(s):     return yx_to_ij(s.to(device), N_VIS)
else:
    def to_ij_signal(s):     return s.to(device)

print(f"coords_full : {tuple(coords_full.shape)}  (SquareSampler ij layout, N={N_VIS}x{N_VIS})")
print(f"yx-layout corpus available : {HAS_YX_DATA}")

### Quick look at the corpus

`function_generator(seed)` returns the data in its native layout. If the
generator stores a regular yx grid (CelebA), we permute through
`yx_to_ij` so the rest of the notebook only sees ij-layout signals.

In [ ]:
fig, axes = plt.subplots(1, N_PEEK, figsize=(2.4 * N_PEEK, 2.4))

# Same call the training loop makes every step.  All N_PEEK samples share one
# randomly-drawn drop pattern (a single CelebA "draw") — exactly what the model
# sees inside a single get_batch call.
coords_b, vals_b = function_generator.get_batch(batch_size=N_PEEK)   # (N', d), (B, N', C)

for j, ax in enumerate(np.atleast_1d(axes)):
    vals_j = vals_b[j]
    if HAS_YX_DATA and vals_j.shape[0] == N_FULL:
        show_field(ax, to_ij_signal(vals_j), N_VIS, title=f"sample {j}", cmap='Greys_r')
    elif vals_j.shape[0] == N_FULL:
        show_field(ax, vals_j.to(device), N_VIS, title=f"sample {j}", cmap='Greys_r')
    else:
        # irregular drop pattern — fall back to scatter (channel-agnostic helper)
        scatter_signal(ax, coords_b, vals_j,
                       title=f"sample {j} (N'={vals_j.shape[0]})", size_c1=15, cmap='Greys_r')

plt.suptitle(f"{type(function_generator).__name__}.get_batch() — what the model sees during training",
             fontsize=10)
plt.tight_layout(); plt.show()

## 3. Pushforward of Fourier atoms — prefix mosaic

Evaluate the first `MOSAIC_SIDE`² atoms $e_1, e_2, \dots$ of the ordered Fourier
prefix (descending PMF) on a regular grid, push the prefix through the
isometry, and lay both sets of tiles out in a `MOSAIC_SIDE` × `MOSAIC_SIDE`
mosaic. Rows hold consecutive prefix positions, starting from the bottom row;
a tile's prefix position is its row label plus its column label.

For $C > 1$ the PMF does not depend on the channel, so the prefix cycles through
the channels at each frequency; every tile is the per-pixel RGB of the atom (the
pushforward generally mixes channels). We render two figures: the original
Fourier atoms $e_k$ and the learned pushforward $T e_k$, followed by an
arbitrary-resolution zoom on one learned atom.

In [ ]:
# Basis visualization controls are in the parameter cell.

# ij-layout grid for tiles (matches field_to_img / to_rgb_image in notebook_helpers).
coords_atom = SquareSampler(stratified=True, add_noise=False).sample(N_VIS_ATOM).to(device)
lad_atom    = torch.zeros(coords_atom.shape[0], device=device)

# Shortest prefix (ν descending) that covers both figures.
n_mosaic    = MOSAIC_SIDE * MOSAIC_SIDE
n_basis_vis = max(N_SHOW_LINEAR, n_mosaic)

with torch.no_grad():
    atoms_orig   = raw_prefix(coords_atom, n_basis_vis)       # (A, N, C)
    atoms_pushed = learned_prefix(coords_atom, n_basis_vis)   # (A, N, C)

# ── 2-row linear comparison (prefix order) ───────────────────────────────────
plot_atoms_comparison(
    atoms_orig.detach().cpu(), atoms_pushed.detach().cpu(),
    n_vis=N_VIS_ATOM, n_show=N_SHOW_LINEAR, tile_size=TILE_SIZE,
    cmap='Greys_r',
    title=f"First {N_SHOW_LINEAR} prefix atoms (ν descending)",
)

# ── Mosaic of the first MOSAIC_SIDE² prefix atoms (both bases) ───────────────
mosaic_idx = prefix_mosaic_indices(MOSAIC_SIDE)
plot_atom_mosaic(
    atoms_orig[:n_mosaic].detach().cpu(), mosaic_idx,
    title=fr"Original Fourier atoms  $e_k$  (first {n_mosaic} prefix atoms)",
    n_vis=N_VIS_ATOM, tile_size=TILE_SIZE,
    cmap='Greys_r',
)
plot_atom_mosaic(
    atoms_pushed[:n_mosaic].detach().cpu(), mosaic_idx,
    title=fr"Learned atoms (pushforward)  $T e_k$  (first {n_mosaic} prefix atoms)",
    n_vis=N_VIS_ATOM, tile_size=TILE_SIZE,
    cmap='Greys_r',
)

# ── Zoom inset on learned atom ZOOM_ATOM_IDX (arbitrary-resolution evaluation) ──
# Important: the Cayley step inside `pushforward` computes inner products as
# `(1/N) Σ_n U(x_n) · f(x_n)`, which only approximates `∫_{[0,1]²} U·f dx`
# when the points cover the full domain uniformly. If we fed pushforward a
# grid restricted to a sub-region, the same `(1/N) Σ_n` would instead average
# over that sub-region — a *different* operator — so the values would not be
# directly comparable to the full-domain pushforward.
#
# Coherent recipe: evaluate T e_k once on a single high-res grid covering all of
# `[0,1]²`, then derive both panels from that array (downsample for the left,
# crop for the right). Same operator, same scale, faithful colours.
# Zoom controls are configured in the parameter cell.

import matplotlib.patches as mpatches

# Pick a high-res evaluation grid such that ZOOM_RES pixels land inside the box.
zoom_side     = max(ZOOM_RIGHT - ZOOM_LEFT, ZOOM_UP - ZOOM_DOWN)
ZOOM_HIGH_RES = max(ZOOM_FULL_RES, int(round(ZOOM_RES / max(zoom_side, 1e-9))))
print(f"High-res Q-evaluation grid: {ZOOM_HIGH_RES}x{ZOOM_HIGH_RES}  "
      f"(box side = {zoom_side:.3f}, → ≈{int(round(ZOOM_HIGH_RES * zoom_side))}px in the zoom)")

# Single full-domain pushforward — both panels are derived from this array.
coords_hi = SquareSampler(stratified=True, add_noise=False).sample(ZOOM_HIGH_RES).to(device)

# Only the prefix up to the zoomed atom is evaluated (lengthened to R inside
# `learned_prefix`); by prefix faithfulness its last row is the same learned atom
# as in the mosaic above.
with torch.no_grad():
    atom_hi_pushed = learned_prefix(coords_hi, ZOOM_ATOM_IDX + 1)[ZOOM_ATOM_IDX:]

# (N², C) → (H_y, W_x, C) using the same ij→display convention as notebook_helpers.
img_hi = (
    atom_hi_pushed[0].detach().cpu().float()
    .reshape(ZOOM_HIGH_RES, ZOOM_HIGH_RES, -1).transpose(0, 1).numpy()
)
C = img_hi.shape[-1]

# Left panel: block-mean downsample to ZOOM_FULL_RES.
bs    = max(1, ZOOM_HIGH_RES // ZOOM_FULL_RES)
n_out = ZOOM_HIGH_RES // bs
img_full = (
    img_hi[: n_out * bs, : n_out * bs]
    .reshape(n_out, bs, n_out, bs, C)
    .mean(axis=(1, 3))
)

# Right panel: crop the zoom box directly from the high-res evaluation.
# Rows = y, cols = x; extent=[0,1,0,1] with origin='lower'.
x_lo = int(round(ZOOM_LEFT  * ZOOM_HIGH_RES))
x_hi = int(round(ZOOM_RIGHT * ZOOM_HIGH_RES))
y_lo = int(round(ZOOM_DOWN  * ZOOM_HIGH_RES))
y_hi = int(round(ZOOM_UP    * ZOOM_HIGH_RES))
img_zoom = img_hi[y_lo:y_hi, x_lo:x_hi]

# Shared normalization across both panels (no per-tile renormalization).
if C == 1:
    vmax = float(max(np.abs(img_full[..., 0]).max(), np.abs(img_zoom[..., 0]).max())) + 1e-8
    vmin = -vmax
    def _draw(ax, img, ext):
        ax.imshow(img[..., 0], origin='lower', extent=ext,
                  cmap='Greys_r', vmin=vmin, vmax=vmax)
else:
    lo = float(min(img_full[..., :3].min(), img_zoom[..., :3].min()))
    hi = float(max(img_full[..., :3].max(), img_zoom[..., :3].max()))
    rng = max(hi - lo, 1e-8)
    def _draw(ax, img, ext):
        x = np.clip((img[..., :3] - lo) / rng, 0.0, 1.0)
        ax.imshow(x, origin='lower', extent=ext)

fig, axes = plt.subplots(1, 2, figsize=(8.4, 4.2))

# Left: full domain at ZOOM_FULL_RES, with a red rectangle marking the zoom region.
_draw(axes[0], img_full, [0, 1, 0, 1])
axes[0].add_patch(mpatches.Rectangle(
    (ZOOM_LEFT, ZOOM_DOWN),
    ZOOM_RIGHT - ZOOM_LEFT, ZOOM_UP - ZOOM_DOWN,
    linewidth=2.0, edgecolor='red', facecolor='none',
))
axes[0].set_xticks([]); axes[0].set_yticks([]); axes[0].set_aspect('equal')
axes[0].set_title(
    fr"$T e_{{{ZOOM_ATOM_IDX + 1}}}$  on  $[0,1]^2$  at  {ZOOM_FULL_RES}$\times${ZOOM_FULL_RES}",
    fontsize=8,
)

# Right: same array, cropped to the zoom box.
_draw(axes[1], img_zoom, [ZOOM_LEFT, ZOOM_RIGHT, ZOOM_DOWN, ZOOM_UP])
axes[1].set_xticks([]); axes[1].set_yticks([]); axes[1].set_aspect('equal')
axes[1].set_title(
    fr"Zoom  $[{ZOOM_LEFT:.2f},{ZOOM_RIGHT:.2f}]\times[{ZOOM_DOWN:.2f},{ZOOM_UP:.2f}]$"
    f"  ({img_zoom.shape[1]}$\\times${img_zoom.shape[0]} px)",
    fontsize=8,
)
for sp in axes[1].spines.values():
    sp.set_edgecolor('red'); sp.set_linewidth(2.0)

plt.suptitle(
    f"Arbitrary-resolution evaluation",
    fontsize=10,
)
plt.tight_layout(); plt.show()


## 4. Mixing layers — functions $u_r$ and causal attention $A$

Each of the `num_layers` layers is one exact Cayley rotation
$Q_\ell = (I-\tfrac12 L_\ell)^{-1}(I+\tfrac12 L_\ell)$ with the skew generator
$L_\ell = \sum_r u_r\langle a_r,\cdot\rangle - a_r\langle u_r,\cdot\rangle$,
$a_r = \sum_{k\le r} A_{rk} e_k$. `visualize_mixing_layers` plots, per layer, the
first mixing functions $u_r(x)$ and the lower-triangular attention $A$.

In [ ]:
coords_vis_U = SquareSampler(stratified=True, add_noise=False).sample(N_VIS_U).to(device)

visualize_mixing_layers(neural_isometry, coords_vis_U, RANK, MIXING_LAYERS, trunc=5)

## 5. Learned mean function $\mu(x)$

The mean function is co-trained with the isometry; for FPCA it minimises
MSE to the dataset mean and the isometry diagonalises the covariance of
the zero-centred field.

In [ ]:
with torch.no_grad():
    mean_vals = mean_function(coords_full)   # (N, C)

fig, ax = plt.subplots(figsize=(4.6, 4.6))
show_field(ax, mean_vals, N_VIS,
           title=fr"Learned mean function  $\mu(x)$  (C={CHANNELS_DIM})", cmap='Greys_r')
plt.tight_layout(); plt.show()

## 6. Reconstruction with the learned basis

For a held-out batch of signals, compare the reconstruction from the first
$k$ atoms of the **Fourier** prefix $\{e_j\}_{j\le k}$ with the reconstruction
from the first $k$ atoms of the **learned** prefix $\{T e_j\}_{j\le k}$, for each
budget $k$ in `RECON_ATOM_BUDGETS`. Both are the orthogonal projection of
$f-\mu$ onto the span of those atoms, computed through their Gram matrix so
the atoms need not be orthonormal. PSNR is shown above each tile.

Validation signals come from `fetch_validation_signals` — for CelebA we
permute through `yx_to_ij` so the reconstruction grid matches `coords_full`.

In [ ]:
val_seed_offset = getattr(function_generator, "n_images", 0) * \
                  getattr(function_generator, "n_drop_draws", 1)

n_atoms_list = list(dict.fromkeys(RECON_ATOM_BUDGETS))
print(f"Atom budgets (prefix lengths) : {n_atoms_list}")

val_signals_raw = fetch_validation_signals(
    function_generator, coords_full, N_VAL, device, val_seed_offset=val_seed_offset,
)
val_signals = to_ij_signal(val_signals_raw)
print(f"Validation signals : {tuple(val_signals.shape)}  (ij layout)")

In [ ]:
# One prefix per basis, long enough for the largest budget; budget k uses its first k rows.
with torch.no_grad():
    mean_full     = mean_function(coords_full)
    basis_fourier = raw_prefix(coords_full, max(n_atoms_list))
    basis_learned = learned_prefix(coords_full, max(n_atoms_list))

    fourier_recons = [reconstruct_in_basis(val_signals, mean_full, basis_fourier[:k]).cpu()
                      for k in n_atoms_list]
    learned_recons = [reconstruct_in_basis(val_signals, mean_full, basis_learned[:k]).cpu()
                      for k in n_atoms_list]
print(f"Computed {len(n_atoms_list)} atom budgets x {N_VAL} signals "
      f"for Fourier and Learned bases.")

In [ ]:
val_signals_cpu = val_signals.cpu()
metrics_part = plot_recon_grid(
    methods=[("Learned", learned_recons), ("Fourier", fourier_recons)],
    val_imgs_cpu=val_signals_cpu, n_atoms_list=n_atoms_list, n=N_VIS,
    cmap='Greys_r',
)
learned_metrics = metrics_part["Learned"]
fourier_metrics = metrics_part["Fourier"]

## 7. Baselines

* **PCA Gold** — classical PCA on the *full-grid* corpus.
* **PCA Interpolated** — classical PCA after interpolating each *sparse*
  observation back to the full grid (only meaningful for generators with
  pre-computed drop patterns; skipped otherwise).

In [ ]:
corpus_raw = build_corpus(function_generator, coords_full).float()  # (M, N, C) in native layout
corpus     = to_ij_signal(corpus_raw).cpu()                          # (M, N, C) in ij layout
M, N, C    = corpus.shape
D          = N * C
X_flat     = corpus.reshape(M, D)
print(f"Corpus shape: {tuple(corpus.shape)}   D = N*C = {D}   (ij layout)")

import os
_run_name  = os.path.basename(os.path.normpath(CHECKPOINT_DIR))
_cache_dir = os.path.normpath(os.path.join(CHECKPOINT_DIR, "..", "..", "notebooks", _run_name))
_cache_gold = os.path.join(_cache_dir, "pca_gold.npz")

if os.path.exists(_cache_gold):
    print(f"[cache hit] Loading PCA gold from {_cache_gold}")
    _d = np.load(_cache_gold)
    mean_face_flat = torch.from_numpy(_d["mean_face_flat"])
    S              = torch.from_numpy(_d["S"])
    Vh             = torch.from_numpy(_d["Vh"])
else:
    print("[cache miss] Computing SVD (may be slow)…")
    mean_face_flat = X_flat.mean(dim=0)
    X_centered     = X_flat - mean_face_flat
    _, S, Vh       = torch.linalg.svd(X_centered, full_matrices=False)
    os.makedirs(_cache_dir, exist_ok=True)
    np.savez_compressed(_cache_gold,
                        mean_face_flat=mean_face_flat.numpy(),
                        S=S.numpy(), Vh=Vh.numpy())
    print(f"[cache saved] {_cache_gold}")

ev = (S ** 2) / (S ** 2).sum()
for k in [1, 5, 10, 20, 50, 100]:
    if k <= len(ev):
        print(f"  Top-{k:>3d} PCs explain {ev[:k].sum().item()*100:5.1f}% of variance")


In [ ]:
K_SHOW = min(K_SHOW, Vh.shape[0])
mean_img        = mean_face_flat.reshape(N, C)
gold_components = Vh[:K_SHOW].reshape(K_SHOW, N, C)

fig, axes = plt.subplots(1, K_SHOW + 1, figsize=(2.4 * (K_SHOW + 1), 2.4))
show_field(axes[0], mean_img, N_VIS, title="dataset mean", cmap='Greys_r')
for i in range(K_SHOW):
    show_field(axes[i + 1], gold_components[i], N_VIS, title=f"PC {i+1}\n$\\sigma$={S[i]:.1f}", cmap='Greys_r')
plt.suptitle(f"PCA Gold — top components from {M} signals on the full {N_VIS}x{N_VIS} grid",
             fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
from scipy.interpolate import griddata as scipy_griddata
from tqdm import tqdm

HAVE_DROP_PATTERNS = (
    hasattr(function_generator, "_drop_patterns") and
    hasattr(function_generator, "_drop_coords") and
    hasattr(function_generator, "images")
)

if HAVE_DROP_PATTERNS:
    import os
    _cache_interp = os.path.join(_cache_dir, "pca_interp.npz")

    if os.path.exists(_cache_interp):
        print(f"[cache hit] Loading PCA interp from {_cache_interp}")
        _d         = np.load(_cache_interp)
        mean_interp = torch.from_numpy(_d["mean_interp"])
        S_i         = torch.from_numpy(_d["S_i"])
        Vh_interp   = torch.from_numpy(_d["Vh_interp"])
        print(f"Vh_interp : {tuple(Vh_interp.shape)}  (ij layout, from cache)")
    else:
        full_coords_yx_np = function_generator._full_coords.detach().cpu().numpy()
        n_total = function_generator.n_images * function_generator.n_drop_draws
        print(f"Drop fraction : {function_generator.drop_fraction*100:.0f}%")
        print(f"Drop draws    : {function_generator.n_drop_draws} per image  ->  {n_total} rows")
        print(f"Pattern sizes : min={min(len(p) for p in function_generator._drop_patterns)}  "
              f"max={max(len(p) for p in function_generator._drop_patterns)} pixels kept")

        def interp_to_full(sparse_coords_np, vals_np):
            filled = scipy_griddata(sparse_coords_np, vals_np, full_coords_yx_np,
                                    method="linear", fill_value=float("nan"))
            nan_mask = np.isnan(filled[:, 0])
            if nan_mask.any():
                filled[nan_mask] = scipy_griddata(sparse_coords_np, vals_np,
                                                  full_coords_yx_np[nan_mask], method="nearest")
            return filled

        interp_signals = []
        for seed in tqdm(range(n_total), desc="Interpolating"):
            sparse_coords, vals = function_generator(seed)
            interp_signals.append(interp_to_full(
                sparse_coords.detach().cpu().numpy(),
                vals.detach().cpu().float().numpy(),
            ))
        Mtot = len(interp_signals)
        corpus_interp_yx = torch.from_numpy(np.stack(interp_signals, axis=0)).float()
        corpus_interp_ij = yx_to_ij(corpus_interp_yx, N_VIS)
        X_interp_flat    = corpus_interp_ij.reshape(Mtot, D)
        mean_interp      = X_interp_flat.mean(dim=0)
        _, S_i, Vh_interp = torch.linalg.svd(X_interp_flat - mean_interp, full_matrices=False)
        print(f"\nVh_interp : {tuple(Vh_interp.shape)}  (ij layout)")

        os.makedirs(_cache_dir, exist_ok=True)
        np.savez_compressed(_cache_interp,
                            mean_interp=mean_interp.numpy(),
                            S_i=S_i.numpy(),
                            Vh_interp=Vh_interp.numpy())
        print(f"[cache saved] {_cache_interp}")

    ev_g = (S    ** 2) / (S    ** 2).sum()
    ev_i = (S_i  ** 2) / (S_i  ** 2).sum()
    print(f"\n{'K':>4} | {'Gold cumvar':>12} | {'Interp cumvar':>14}")
    print("-" * 40)
    for k in [1, 5, 10, 20, 50]:
        if k <= len(S):
            print(f"{k:>4} | {ev_g[:k].sum()*100:>11.1f}% | {ev_i[:k].sum()*100:>13.1f}%")
else:
    print(f"Generator {type(function_generator).__name__} has no sparse drop "
          "patterns — skipping the interpolated-PCA baseline.")
    Vh_interp, mean_interp = None, None


In [ ]:
with torch.no_grad():
    pca_gold_recons = pca_reconstruct(
        val_signals, mean_face_flat, Vh, n_atoms_list, N_FULL, CHANNELS_DIM, device,
    )
    pca_interp_recons = (
        pca_reconstruct(val_signals, mean_interp, Vh_interp, n_atoms_list,
                         N_FULL, CHANNELS_DIM, device)
        if Vh_interp is not None else None
    )

methods_for_grid = [("Learned",  learned_recons),
                    ("Fourier",  fourier_recons),
                    ("PCA Gold", pca_gold_recons)]
if pca_interp_recons is not None:
    methods_for_grid.append((f"PCA Interp ({function_generator.drop_fraction*100:.0f}% dropped)",
                             pca_interp_recons))

all_metrics = plot_recon_grid(
    methods=methods_for_grid, val_imgs_cpu=val_signals_cpu,
    n_atoms_list=n_atoms_list, n=N_VIS, cmap='Greys_r'
)

## 8. PSNR vs number of atoms

One subplot per validation signal, bars grouped by truncation level.

In [ ]:
methods = [
    ("Learned",  all_metrics["Learned"]),
    ("Fourier",  all_metrics["Fourier"]),
    # ("PCA Gold", all_metrics["PCA Gold"]),
]
# if pca_interp_recons is not None:
#     methods.append(("PCA Interp",
#                     all_metrics[f"PCA Interp ({function_generator.drop_fraction*100:.0f}% dropped)"]))

L         = val_signals_cpu.shape[0]
M_methods = len(methods)
bar_w     = 0.8 / M_methods
xs        = np.arange(len(n_atoms_list))
colors    = plt.cm.tab10(np.arange(M_methods))

fig, axes = plt.subplots(1, L, figsize=(4.4 * L, 4.2), sharey=True, squeeze=False)
for l, ax in enumerate(axes[0]):
    for m_idx, (method_name, metrics) in enumerate(methods):
        psnrs   = [pp for (img, _, pp) in metrics if img == l]
        offsets = (np.arange(M_methods) - (M_methods - 1) / 2) * bar_w
        ax.bar(xs + offsets[m_idx], psnrs, width=bar_w,
               color=colors[m_idx], label=method_name)
    ax.set_xticks(xs); ax.set_xticklabels(n_atoms_list)
    ax.set_xlabel("Number of atoms")
    ax.set_title(f"signal {l}")
    ax.grid(axis="y", alpha=0.3)
    if l == 0:
        ax.set_ylabel("PSNR (dB)")
    if l == L - 1:
        ax.legend(title="method", fontsize=8, title_fontsize=8)
plt.suptitle("PSNR vs number of atoms", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:

# ── In-distribution: all training images ─────────────────────────────────
N_IN = function_generator.n_images if hasattr(function_generator, "images") else 50

val_in_raw = fetch_validation_signals(
    function_generator, coords_full, N_IN, device, val_seed_offset=val_seed_offset,
)
val_in     = to_ij_signal(val_in_raw)
val_in_cpu = val_in.cpu()
L_in       = val_in.shape[0]
print(f"In-distribution signals : {tuple(val_in.shape)}")

# ── Out-of-distribution: load next images beyond the training split ────────
if hasattr(function_generator, "images"):
    import torchvision.transforms as TF
    from datasets import load_dataset as hf_load_dataset

    _res   = function_generator.resolution
    _N_pix = _res * _res
    _tfs   = [TF.Resize((_res, _res), interpolation=TF.InterpolationMode.BICUBIC)]
    if function_generator.grayscale:
        _tfs.append(TF.Grayscale(num_output_channels=1))
    _tfs.append(TF.ToTensor())
    _to_tensor = TF.Compose(_tfs)

    N_OOD = N_IN   # same size as in-distribution for a fair comparison
    print(f"Loading {N_OOD} OOD images (skipping first {N_IN} from shuffled train split)…")
    _ds  = hf_load_dataset("mattymchen/celeba-hq", split="train", streaming=True)
    # Must use the identical shuffle as CelebADataset.__init__ so that
    # positions 0..N_IN-1 of this stream == training images and
    # positions N_IN..2*N_IN-1 == true OOD (non-overlapping, same distribution).
    # This also ensures pca_reconstruct(val_ood, ...) inside _all_recons
    # operates on correctly-shuffled OOD images.
    _ds  = _ds.shuffle(seed=42, buffer_size=10_000)
    _ood = []
    for _i, _sample in enumerate(_ds):
        if _i < N_IN:
            continue
        if len(_ood) >= N_OOD:
            break
        _t = _to_tensor(_sample["image"]).permute(1, 2, 0).flip(0)   # (H, W, C)
        _ood.append(_t.reshape(_N_pix, function_generator.channels))

    val_ood_yx  = torch.stack(_ood, dim=0).to(device)          # yx layout
    val_ood     = yx_to_ij(val_ood_yx, _res)                   # ij layout
    val_ood_cpu = val_ood.cpu()
    L_ood       = val_ood.shape[0]
    print(f"OOD signals           : {tuple(val_ood.shape)}")
    has_ood = True
else:
    has_ood = False
    print("Generator is not CelebA-style — OOD loading skipped.")

# ── Compute all reconstructions ───────────────────────────────────────────

def _all_recons(val_signals):
    fourier_r, learned_r = [], []
    with torch.no_grad():
        for k in n_atoms_list:
            fourier_r.append(reconstruct_in_basis(val_signals, mean_full, basis_fourier[:k]).cpu())
            learned_r.append(reconstruct_in_basis(val_signals, mean_full, basis_learned[:k]).cpu())
        pca_gold_r = pca_reconstruct(
            val_signals, mean_face_flat, Vh, n_atoms_list, N_FULL, CHANNELS_DIM, device,
        )
        pca_interp_r = (
            pca_reconstruct(val_signals, mean_interp, Vh_interp, n_atoms_list, N_FULL, CHANNELS_DIM, device)
            if Vh_interp is not None else None
        )
    return fourier_r, learned_r, pca_gold_r, pca_interp_r

fourier_in, learned_in, pca_gold_in, pca_interp_in = _all_recons(val_in)
if has_ood:
    fourier_ood, learned_ood, pca_gold_ood, pca_interp_ood = _all_recons(val_ood)
print("Reconstructions complete.")

# ── PSNR matrix helper ────────────────────────────────────────────────────
def _build_method_pms(fourier_r, learned_r, pca_gold_r, pca_interp_r, ref_cpu):
    pms = [
        ("Learned",  psnr_matrix(learned_r,  ref_cpu), plt.cm.tab10(0)),
        ("Fourier",  psnr_matrix(fourier_r,  ref_cpu), plt.cm.tab10(1)),
        # ("PCA Gold", psnr_matrix(pca_gold_r, ref_cpu), plt.cm.tab10(2)),
    ]
    # if pca_interp_r is not None:
    #     drop_pct = f"{function_generator.drop_fraction * 100:.0f}%"
    #     pms.append((f"PCA Interp ({drop_pct} dropped)",
    #                 psnr_matrix(pca_interp_r, ref_cpu), plt.cm.tab10(3)))
    return pms

pms_in  = _build_method_pms(fourier_in,  learned_in,  pca_gold_in,  pca_interp_in,  val_in_cpu)
pms_ood = (_build_method_pms(fourier_ood, learned_ood, pca_gold_ood, pca_interp_ood, val_ood_cpu)
           if has_ood else None)

# ── Side-by-side PSNR line plots ─────────────────────────────────────────
plot_psnr_line_chart(
    in_methods=pms_in,
    n_atoms_list=n_atoms_list,
    ood_methods=pms_ood if has_ood else None,
)

# ── OOD reconstruction grid (4 images) ───────────────────────────────────
if has_ood:
    val_ood_show = val_ood[:N_OOD_SHOW].cpu()
    ood_grid_methods = [
        ("Learned",  [r[:N_OOD_SHOW] for r in learned_ood]),
        ("Fourier",  [r[:N_OOD_SHOW] for r in fourier_ood]),
        ("PCA Gold", [r[:N_OOD_SHOW] for r in pca_gold_ood]),
    ]
    if pca_interp_ood is not None:
        drop_pct = f"{function_generator.drop_fraction * 100:.0f}%"
        ood_grid_methods.append(
            (f"PCA Interp ({drop_pct} dropped)", [r[:N_OOD_SHOW] for r in pca_interp_ood])
        )
    plot_recon_grid(
        methods=ood_grid_methods,
        val_imgs_cpu=val_ood_show,
        n_atoms_list=n_atoms_list,
        n=N_VIS,
        cmap='Greys_r',
    )


## 9. Spectral compactness — power-law spectrum of Fourier vs. learned basis

For each image $f$ in a held-out batch, measure how much of the energy
$\|f-\mu\|^2$ lies in the span of the first $k$ atoms of the ordered prefix in
both bases ($\nu(k)$, the prior PMF, descending), for every $k \le$ `SPECTRAL_ATOMS`:

$$E_k^F = \|P_{\langle e_1,\dots,e_k\rangle}(f - \mu)\|^2, \qquad
  E_k^L = \|P_{\langle T e_1,\dots,T e_k\rangle}(f - \mu)\|^2,$$

where $T e_k$ is row $k$ of the pushed prefix on the same grid. All $k$ come
from one Gram–Schmidt pass in prefix order, so the prefix need not be
orthonormal. The curves show $E_k/\|f-\mu\|^2$; the per-atom panel shows the
increment $E_k - E_{k-1}$, the energy atom $k$ adds beyond atoms $1..k-1$
(for an orthonormal prefix, such as Fourier, this is $\langle f-\mu, e_k\rangle^2$).
If the FPCA model is well-trained, the learned basis packs more energy into
the first few atoms — the learned curve should **start higher** and **decay
faster** than the Fourier curve. We also print the training objective: the
PMF-weighted energy $\sum_{k\le K}\nu(k)\langle f-\mu, T e_k\rangle^2$ over
the $K=$ `num_atoms` prefix.

In [ ]:
# Spectral controls are configured in the parameter cell.

# First SPECTRAL_ATOMS atoms of the ordered prefix (PMF ν descending)
A_spec  = SPECTRAL_ATOMS
spec_nu = initial_dictionary.get_prefix_pmfs(A_spec).cpu()                # (A,)
print(f"{A_spec} prefix atoms, ν ∈ [{spec_nu[-1]:.2e}, {spec_nu[0]:.2e}]")

# Build image batch — reuse corpus if section 7 already ran, else sample fresh
try:
    imgs = to_ij_signal(corpus_raw[:N_SPEC].float().to(device))
except NameError:
    imgs = to_ij_signal(
        fetch_validation_signals(function_generator, coords_full, N_SPEC, device)
    )
imgs = imgs[:N_SPEC].to(device)   # (M, N, C)
M_spec = imgs.shape[0]
print(f"Using {M_spec} images")

In [ ]:
with torch.no_grad():
    mu       = mean_function(coords_full)           # (N, C)
    centered = imgs - mu.unsqueeze(0)               # (M, N, C)
    lad      = logabsdet_full                       # (N,)

    # One prefix per basis, covering both the spectrum and the training objective.
    n_spec_prefix  = max(A_spec, NUM_ATOMS)
    prefix_raw     = raw_prefix(coords_full, n_spec_prefix)       # e_1..e_n
    prefix_learned = learned_prefix(coords_full, n_spec_prefix)   # T e_1..T e_n

    # E_k = ‖P_k (f - μ)‖² for every prefix length k <= A, in the Fourier and learned bases, and ‖f - μ‖².
    E_F = nested_projection_energies(centered, prefix_raw[:A_spec], lad).cpu()       # (M, A)
    E_L = nested_projection_energies(centered, prefix_learned[:A_spec], lad).cpu()   # (M, A)
    tot = norm2(centered, lad).cpu()                                                 # (M,)

    # Training objective: PMF-weighted energy over the K-atom prefix.
    E_prefix_F = prefix_weighted_energy(centered, prefix_raw[:NUM_ATOMS], PREFIX_PMFS, lad).mean()
    E_prefix_L = prefix_weighted_energy(centered, prefix_learned[:NUM_ATOMS], PREFIX_PMFS, lad).mean()

# Energy atom k adds beyond atoms 1..k-1 (⟨f - μ, e_k⟩² for an orthonormal prefix).
dE_F = torch.diff(E_F, dim=-1, prepend=torch.zeros_like(E_F[:, :1]))   # (M, A)
dE_L = torch.diff(E_L, dim=-1, prepend=torch.zeros_like(E_L[:, :1]))
print(f"Total energy ‖f-μ‖²: {tot.mean():.4f}")
print(f"Energy in the first {A_spec} atoms — Fourier: {E_F[:, -1].mean():.4f}  |  Learned: {E_L[:, -1].mean():.4f}")
print(f"PMF-weighted energy (training objective, K={NUM_ATOMS}) — Fourier: {E_prefix_F:.4e}  |  Learned: {E_prefix_L:.4e}")

In [ ]:
import numpy as np


# Per-image cumulative fraction of total energy ‖P_k f‖² / ‖f‖²: (M, A)
cum_F_per_img = E_F / (tot[:, None] + 1e-15)
cum_L_per_img = E_L / (tot[:, None] + 1e-15)

cum_F_mean = cum_F_per_img.mean(dim=0).numpy()    # (A,)
cum_F_std  = cum_F_per_img.std(dim=0).numpy()
cum_L_mean = cum_L_per_img.mean(dim=0).numpy()
cum_L_std  = cum_L_per_img.std(dim=0).numpy()

x = np.arange(1, A_spec + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

plot_spectral_compactness(
    [
        dict(label="Fourier", xs=x, mean=cum_F_mean, std=cum_F_std, color='C0'),
        dict(label="Learned", xs=x, mean=cum_L_mean, std=cum_L_std, color='C3'),
    ],
    xlabel="atom rank  (ν descending)",
    xscale="log",
    axes=(axes[0], axes[1]),
)

# ── 3. Per-atom mean ± std  (top N_VIOLIN atoms) ─────────────────────────
ax  = axes[2]
nv  = min(N_VIOLIN, A_spec)
pos = np.arange(1, nv + 1)

mean_F = dE_F[:, :nv].mean(dim=0).numpy()   # (nv,)
std_F  = dE_F[:, :nv].std(dim=0).numpy()
mean_L = dE_L[:, :nv].mean(dim=0).numpy()
std_L  = dE_L[:, :nv].std(dim=0).numpy()

ax.plot(pos, mean_F, 'o-', lw=1.5, ms=4, color='C0', label="Fourier")
ax.fill_between(pos, np.maximum(mean_F - std_F, 0), mean_F + std_F,
                alpha=0.25, color='C0')
ax.plot(pos, mean_L, 'o-', lw=1.5, ms=4, color='C3', label="Learned")
ax.fill_between(pos, np.maximum(mean_L - std_L, 0), mean_L + std_L,
                alpha=0.25, color='C3')
ax.set_xticks(pos)
ax.set_xticklabels([str(k) for k in pos], fontsize=7)
ax.set_xlabel("atom rank  (ν descending, with tiebreak)")
ax.set_ylabel("$E_k - E_{k-1}$  (mean ± 1 std)")
ax.set_title(f"Per-image energy  (top {nv} atoms)")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")

plt.suptitle(
    f"Spectral compactness — Fourier vs. Learned basis  "
    f"({M_spec} images, first {A_spec} prefix atoms)",
    y=1.02,
)
plt.tight_layout()
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────
print_threshold_table(cum_F_mean, cum_L_mean)

In [ ]:

# ── Spectral compactness on OOD images ────────────────────────────────────
# Reuses prefix_raw / prefix_learned / lad / A_spec / N_VIOLIN from section 9.
# Requires the OOD cell (section 8) to have run first.

if has_ood:

    with torch.no_grad():
        mu_ood       = mean_function(coords_full)                                # (N, C)
        centered_ood = val_ood.to(device) - mu_ood.unsqueeze(0)                 # (L_ood, N, C)

        E_F_ood = nested_projection_energies(centered_ood, prefix_raw[:A_spec], lad).cpu()      # (L_ood, A)
        E_L_ood = nested_projection_energies(centered_ood, prefix_learned[:A_spec], lad).cpu()  # (L_ood, A)
        tot_ood = norm2(centered_ood, lad).cpu()                                                # (L_ood,)
        E_prefix_F_ood = prefix_weighted_energy(
            centered_ood, prefix_raw[:NUM_ATOMS], PREFIX_PMFS, lad).mean()
        E_prefix_L_ood = prefix_weighted_energy(
            centered_ood, prefix_learned[:NUM_ATOMS], PREFIX_PMFS, lad).mean()

    dE_F_ood = torch.diff(E_F_ood, dim=-1, prepend=torch.zeros_like(E_F_ood[:, :1]))   # (L_ood, A)
    dE_L_ood = torch.diff(E_L_ood, dim=-1, prepend=torch.zeros_like(E_L_ood[:, :1]))
    print(f"OOD total energy ‖f-μ‖²: {tot_ood.mean():.4f}")
    print(f"OOD energy in the first {A_spec} atoms — Fourier: {E_F_ood[:, -1].mean():.4f}  |  "
          f"Learned: {E_L_ood[:, -1].mean():.4f}")
    print(f"OOD PMF-weighted energy (training objective, K={NUM_ATOMS}) — Fourier: {E_prefix_F_ood:.4e}  |  "
          f"Learned: {E_prefix_L_ood:.4e}")

    # Per-image cumulative fraction of total energy ‖P_k f‖² / ‖f‖²: (L_ood, A)
    cum_F_ood_per_img = E_F_ood / (tot_ood[:, None] + 1e-15)
    cum_L_ood_per_img = E_L_ood / (tot_ood[:, None] + 1e-15)

    cum_F_ood_mean = cum_F_ood_per_img.mean(dim=0).numpy()
    cum_F_ood_std  = cum_F_ood_per_img.std(dim=0).numpy()
    cum_L_ood_mean = cum_L_ood_per_img.mean(dim=0).numpy()
    cum_L_ood_std  = cum_L_ood_per_img.std(dim=0).numpy()

    x_ood = np.arange(1, A_spec + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

    plot_spectral_compactness(
        [
            dict(label="Fourier", xs=x_ood, mean=cum_F_ood_mean, std=cum_F_ood_std, color='C0'),
            dict(label="Learned", xs=x_ood, mean=cum_L_ood_mean, std=cum_L_ood_std, color='C3'),
        ],
        xlabel="atom rank  (ν descending)",
        xscale="log",
        axes=(axes[0], axes[1]),
    )

    # ── Per-atom mean ± std  (top N_VIOLIN atoms) ─────────────────────────
    ax  = axes[2]
    nv  = min(N_VIOLIN, A_spec)
    pos = np.arange(1, nv + 1)

    mean_F_ood = dE_F_ood[:, :nv].mean(dim=0).numpy()
    std_F_ood  = dE_F_ood[:, :nv].std(dim=0).numpy()
    mean_L_ood = dE_L_ood[:, :nv].mean(dim=0).numpy()
    std_L_ood  = dE_L_ood[:, :nv].std(dim=0).numpy()

    ax.plot(pos, mean_F_ood, 'o-', lw=1.5, ms=4, color='C0', label="Fourier")
    ax.fill_between(pos, np.maximum(mean_F_ood - std_F_ood, 0), mean_F_ood + std_F_ood,
                    alpha=0.25, color='C0')
    ax.plot(pos, mean_L_ood, 'o-', lw=1.5, ms=4, color='C3', label="Learned")
    ax.fill_between(pos, np.maximum(mean_L_ood - std_L_ood, 0), mean_L_ood + std_L_ood,
                    alpha=0.25, color='C3')
    ax.set_xticks(pos)
    ax.set_xticklabels([str(k) for k in pos], fontsize=7)
    ax.set_xlabel("atom rank  (ν descending, with tiebreak)")
    ax.set_ylabel("$E_k - E_{k-1}$  (mean ± 1 std)")
    ax.set_title(f"Per-image energy  (top {nv} atoms)")
    ax.legend(); ax.grid(True, alpha=0.3, axis="y")

    plt.suptitle(
        f"Spectral compactness — OOD images  "
        f"({L_ood} images, first {A_spec} prefix atoms)",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    # ── Summary table ─────────────────────────────────────────────────────────
    print_threshold_table(cum_F_ood_mean, cum_L_ood_mean)